 
# Video Sampling Notebook

**Input:** A folder containing `.mp4` video files  
**Output:** One subfolder per video, each containing sampled frames  

**Variable:** Sampling rate (frames per second)


In [1]:
import os
import cv2
import numpy as np
from datetime import timedelta


In [2]:
SAVING_FRAMES_PER_SECOND = 1

In [4]:

def format_timedelta(td):
    result = str(td)
    try:
        result, ms = result.split(".")
    except ValueError:
        return (result + ".00").replace(":", "-")
    ms = int(ms)
    ms = round(ms / 1e4)
    return f"{result}.{ms:02}".replace(":", "-")

def get_saving_frames_durations(cap, saving_fps):
    s = []
    clip_duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
    for i in np.arange(0, clip_duration, 1 / saving_fps):
        s.append(i)
    return s

def process_video(video_file, output_directory):
    video_name = os.path.splitext(os.path.basename(video_file))[0]
    video_output_dir = os.path.join(output_directory, video_name + "-echantiollonee")
    
    if not os.path.isdir(video_output_dir):
        os.mkdir(video_output_dir)
    
    cap = cv2.VideoCapture(video_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    saving_frames_per_second = min(fps, SAVING_FRAMES_PER_SECOND)
    saving_frames_durations = get_saving_frames_durations(cap, saving_frames_per_second)
    count = 0
    
    while True:
        is_read, frame = cap.read()
        if not is_read:
            break
        frame_duration = count / fps
        try:
            closest_duration = saving_frames_durations[0]
        except IndexError:
            break
        if frame_duration >= closest_duration:
            frame_duration_formatted = format_timedelta(timedelta(seconds=frame_duration))
            image_filename = f"{video_name}_frame{frame_duration_formatted}.jpg"
            cv2.imwrite(os.path.join(video_output_dir, image_filename), frame)
            try:
                saving_frames_durations.pop(0)
            except IndexError:
                pass
        count += 1

def main(input_directory, output_directory):
    if not os.path.isdir(output_directory):
        os.mkdir(output_directory)
    
    for filename in os.listdir(input_directory):
        process_video(os.path.join(input_directory, filename), output_directory)




Faster results :  multi-threading

In [3]:
import os
import cv2
import numpy as np
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

SAVING_FRAMES_PER_SECOND = 8  
def format_timedelta(td):
    result = str(td)
    try:
        result, ms = result.split(".")
    except ValueError:
        return (result + ".00").replace(":", "-")
    ms = int(ms)
    ms = round(ms / 1e4)
    return f"{result}.{ms:02}".replace(":", "-")

def get_saving_frames_durations(cap, saving_fps):
    s = []
    clip_duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
    for i in np.arange(0, clip_duration, 1 / saving_fps):
        s.append(i)
    return s

def process_video(video_file, output_directory):
    video_name = os.path.splitext(os.path.basename(video_file))[0]
    video_output_dir = os.path.join(output_directory, video_name + "-echantiollonee")
    
    if not os.path.isdir(video_output_dir):
        os.mkdir(video_output_dir)
    
    cap = cv2.VideoCapture(video_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    saving_frames_per_second = min(fps, SAVING_FRAMES_PER_SECOND)
    saving_frames_durations = get_saving_frames_durations(cap, saving_frames_per_second)
    count = 0
    
    while True:
        is_read, frame = cap.read()
        if not is_read:
            break
        frame_duration = count / fps
        try:
            closest_duration = saving_frames_durations[0]
        except IndexError:
            break
        if frame_duration >= closest_duration:
            frame_duration_formatted = format_timedelta(timedelta(seconds=frame_duration))
            image_filename = f"{video_name}_frame{frame_duration_formatted}.jpg"
            cv2.imwrite(os.path.join(video_output_dir, image_filename), frame)
            try:
                saving_frames_durations.pop(0)
            except IndexError:
                pass
        count += 1

def main(input_directory, output_directory):
    if not os.path.isdir(output_directory):
        os.mkdir(output_directory)
    
    video_files = [os.path.join(input_directory, filename) for filename in os.listdir(input_directory)]

    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_video, video_file, output_directory) for video_file in video_files]
        
        for future in as_completed(futures):
            future.result()  

if __name__ == "__main__":
    input_directory = ""
    output_directory = ""
    main(input_directory, output_directory)
